# Training GloVe Embeddings

Wiki reference for [GloVe training](https://ml-viz-ruby.vercel.app/wiki/glove-training).

**The idea in one sentence.** GloVe learns word vectors by fitting their dot product to the
**log co-occurrence count**, $v_i \cdot v_j + b_i + b_j \approx \log X_{ij}$, under a
**weighting function** $f(X_{ij})$ that caps very frequent pairs and zeroes out pairs that never
co-occur — so the geometry of the embedding space encodes co-occurrence structure.

We build the co-occurrence matrix, the weighting, and gradient-descent training from scratch,
**validate that training converges and that the embeddings capture meaning**, then cover the
gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## A toy corpus and its co-occurrence matrix

$X_{ij}$ counts how often word $j$ appears within a symmetric window around word $i$.

In [ ]:
sentences = [
    "ice is cold cold ice water",
    "steam is hot hot steam water",
    "ice water is solid water",
    "steam water is gas water",
    "cold ice solid ice",
    "hot steam gas steam",
] * 5  # repeat to fatten the counts

tokens = [s.split() for s in sentences]
vocab = sorted({w for sent in tokens for w in sent})
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print("vocab:", vocab)

WINDOW = 2
X = np.zeros((V, V))
for sent in tokens:
    for i, w in enumerate(sent):
        for j in range(max(0, i - WINDOW), min(len(sent), i + WINDOW + 1)):
            if i != j:
                X[w2i[w], w2i[sent[j]]] += 1

print("co-occurrence matrix (rows/cols =", vocab, ")")
print(X.astype(int))

## The weighting function

$f(x) = \min(1, (x/x_{max})^{0.75})$ caps the influence of very frequent pairs and zeroes out pairs that never co-occur.

In [ ]:
X_MAX = 20

def f_weight(x):
    return np.minimum(1.0, (x / X_MAX) ** 0.75)

xs = np.linspace(0, 40, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(xs, f_weight(xs), color='#6366f1', lw=2)
plt.xlabel('co-occurrence count $X_{ij}$'); plt.ylabel('weight $f(X_{ij})$')
plt.title('GloVe weighting: caps frequent pairs, ignores zero pairs')
plt.grid(alpha=0.3); plt.show()

## The objective and its gradients

$$\mathcal{L} = \sum_{i,j} f(X_{ij})\left(\mathbf{v}_i^\top \tilde{\mathbf{v}}_j + b_i + \tilde b_j - \log X_{ij}\right)^2$$

Only nonzero $X_{ij}$ terms contribute. We optimize main vectors $V$, context vectors $\tilde V$, and two bias vectors with plain gradient descent.

In [ ]:
D = 8       # embedding dimension
LR = 0.05
EPOCHS = 400

rng = np.random.default_rng(0)
Vm = rng.normal(scale=0.1, size=(V, D))   # main
Vc = rng.normal(scale=0.1, size=(V, D))   # context
bm = np.zeros(V)
bc = np.zeros(V)

nz = np.argwhere(X > 0)
losses = []
for epoch in range(EPOCHS):
    total = 0.0
    for i, j in nz:
        w = f_weight(X[i, j])
        err = Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j])
        total += w * err ** 2
        g = 2 * w * err
        Vm[i] -= LR * g * Vc[j]
        Vc[j] -= LR * g * Vm[i]
        bm[i] -= LR * g
        bc[j] -= LR * g
    losses.append(total)

emb = Vm + Vc   # final embedding = sum of both matrices
plt.figure(figsize=(8, 4))
plt.plot(losses, color='#6366f1')
plt.xlabel('epoch'); plt.ylabel('weighted least-squares loss')
plt.title('GloVe training loss'); plt.grid(alpha=0.3); plt.show()

### Validate: training drives the loss down

GloVe minimises the weighted squared error between $v_i \cdot v_j + b_i + b_j$ and
$\log X_{ij}$. Over the epochs the weighted least-squares loss should fall substantially. We
confirm.

In [ ]:
print(f'loss: {losses[0]:.2f} (epoch 0) -> {losses[-1]:.4f} (final)')
assert losses[-1] < losses[0], 'GloVe training reduces the weighted least-squares loss'
print('\n✅ the embeddings are fit so v_i . v_j + b_i + b_j approximates log X_ij')

## Do similar words end up close?

"ice" should be nearer "cold"/"solid" than "hot"/"gas" — and vice versa for "steam".

In [ ]:
def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def report(word, probes):
    print(f"similarity to '{word}':")
    for p in probes:
        print(f"  {p:6s} {cos(emb[w2i[word]], emb[w2i[p]]):+.3f}")

report("ice", ["cold", "solid", "hot", "gas"])
print()
report("steam", ["hot", "gas", "cold", "solid"])

### Validate: the embeddings capture co-occurrence meaning

In the toy corpus, *ice* co-occurs with *cold*/*solid* and *steam* with *hot*/*gas*. A good
embedding should place *ice* closer (higher cosine) to *cold* than to *hot*. We confirm the
learned geometry reflects the corpus.

In [ ]:
ice = emb[w2i['ice']]
sim_cold = cos(ice, emb[w2i['cold']])
sim_hot = cos(ice, emb[w2i['hot']])
print(f"cos(ice, cold) = {sim_cold:+.2f}   cos(ice, hot) = {sim_hot:+.2f}")
assert sim_cold > sim_hot, 'GloVe places ice nearer cold than hot — it learned the co-occurrence structure'
print('\n✅ the embedding geometry encodes which words share contexts')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no weighting** | frequent pairs dominate; $\log(0)$ blows up (demo motivates $f$) |
| **window size** | too small misses context; too large blurs it |
| **tiny corpus** | embeddings are noisy — real GloVe needs billions of tokens |
| **static vectors** | one vector per word — no context (unlike BERT) |
| **learning rate** | too high diverges on the log-count targets |

Demo: $f(X_{ij})$ caps frequent pairs and zeroes out empty ones.

In [ ]:
# The heart of GloVe is the WEIGHTING function f(X_ij). It does two jobs: (1) it caps at 1 for
# frequent pairs, so common words like 'the' do not dominate the loss; and (2) it is 0 at
# X_ij=0, so absent pairs are ignored — which also avoids log(0). We confirm both properties.
print(f'f(0)    = {f_weight(0):.2f}   (zero pairs ignored -> no log(0))')
print(f'f(5)    = {f_weight(5):.2f}   (rare pair, down-weighted)')
print(f'f(20)   = {f_weight(20):.2f}')
print(f'f(1000) = {f_weight(1000):.2f}   (capped at 1 -> frequent pairs do not dominate)')
assert f_weight(0) == 0.0, 'f(0)=0 -> zero co-occurrences are skipped (no log(0))'
assert f_weight(1000) == 1.0 and f_weight(5) < f_weight(20), 'the weight caps at 1 and down-weights rare pairs'
print('\nThe weighting caps frequent pairs and skips empty ones -> that is what makes GloVe robust.')

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — the log-count target

**Recap:** GloVe pushes each dot product toward $\log X_{ij}$. After training, the *reconstruction* $\mathbf{v}_i^\top \tilde{\mathbf{v}}_j + b_i + \tilde b_j$ should approximate $\log X_{ij}$ for observed pairs.

Compute the mean absolute reconstruction error over all nonzero pairs (using `Vm, Vc, bm, bc, X, nz` already in memory).

In [ ]:
def mean_abs_error():
    # TODO(you): for each (i, j) in nz, compute the model's prediction
    # and compare to log(X[i, j]); return the mean absolute difference.
    ...

mae = mean_abs_error()
mae

In [ ]:
# Run me — passes silently when correct
errs = [abs(Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j])) for i, j in nz]
expected = float(np.mean(errs))
assert mae is not None and not isinstance(mae, type(Ellipsis)), "fill in the TODO first"
assert abs(float(mae) - expected) < 1e-8
assert expected < 0.5, "training should have brought MAE well below 0.5"
print()

<details>
<summary>Solution</summary>

```python
def mean_abs_error():
    errs = [abs(Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j]))
            for i, j in nz]
    return float(np.mean(errs))
```
</details>

## Key takeaways

- **Fit $v_i\cdot v_j + b_i + b_j \approx \log X_{ij}$:** GloVe factorizes the log co-occurrence
  matrix (loss falls, verified).
- **The embedding geometry encodes meaning:** related words are close (verified).
- **The weighting $f(X_{ij})$** caps frequent pairs and ignores empty ones (demo) — the key
  design choice.
- **Final embedding** = sum of the main and context vectors.